# Deep Learning 基礎講座　最終課題: VQA

## 概要
画像と質問から，回答を予測するタスクです．
- サンプル数: 訓練 19,873 サンプル，テスト 4,969 サンプル
- 入力: 画像データ（RGB，サイズは画像によって異なります），質問文（系列長はサンプルごとに異なります）
- 出力: 回答文（系列長はサンプルごとに異なります）
- 評価指標: VQA での評価指標（[こちら
](https://visualqa.org/evaluation.html)を参照）を利用しています．

### データセット ([VizWiz 2023 edition](https://www.kaggle.com/datasets/nqa112/vizwiz-2023-edition)) の詳細
- 24,842 枚の画像データセットと，各画像に対する 1 つの質問文と 10 人の回答者による回答文から構成されます．
  - 10 人の回答は全て同じとは限りません．
- 24.842 サンプルのうち，80 % (19.873) が訓練データ (train)，20 % (4969) がテストデータ (val) として与えられます．
  - テストデータに対する回答文を正解ラベルとし，配布していません．
  - データ提供元とは異なるデータ分割になっています．

### タスクの詳細
- 本コンペティションでは，与えられた画像と質問文に対して，適切な回答文を出力するモデルを作成していただきます．
- 評価は [VQA](https://visualqa.org/index.html) (Visual Question Answering) に基づいて，以下の式で計算されます．

$$\text{Acc}(ans) = \text{min}(\frac{humans \; that \; said \; ans}{3}, 1)$$

- 1 つのデータに対し， 10 人の回答のうち 9 人の回答を選択し上記の式で性能評価した， 10 パターンの Acc の平均をそのデータに対する Acc とします．
- 予測結果と正解ラベルを比較する前に，回答を lowercase にする，冠詞は削除するなどの前処理を行っています（[詳細](https://visualqa.org/evaluation.html)）．

## 考えられる工夫の例
- 事前学習モデルの fine-tuning
    - 画像特徴量，言語特徴量を取得するときに，事前学習モデルを fine-tuning することで性能向上が見込めます（今回のタスクと大きく異なるデータセットでの事前学習では効果が小さい可能性がありますので注意しましょう）．
- 質問文の表現
    - ベースラインでは，質問文をモデルに入力する際に，one-hot ベクトルにしています．これを tokenizer 等を利用して分散表現にすることで，モデル学習しやすくなります．
- ソフトラベルの利用
    - ベースラインでは 10 人の回答の中で最も多かった回答を正解ラベルとして訓練しています．この点を各回答の頻度に合わせてソフトラベルを利用することで，より多くの情報を利用して学習が可能になります．
- 画像の前処理
    - 画像の前処理には形状を同じにする Resize のみを利用しています．「畳み込みニューラルネットワーク」，「深層学習と画像認識」等で紹介されていたデータ拡張を追加することで，汎化性能の向上が見込めます．

## 修了要件を満たす条件
- ベースラインでは，omnicampus 上での性能評価において， 49.4% となります．したがって，ベースラインを超える 49.4% を超えた提出のみ，修了要件として認めます．
- ベースラインから改善を加えることで， 60% に性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

omnicampus 演習環境では，data_download.ipynb のマウント，zip 化，drive へのコピーを実行しないことで，"data.zip" を解凍した形で配置されます．したがって，data ディレクトリが存在するディレクトリをカレントディレクトリとするだけで良いです．



In [ ]:
# omnicampus 実行用
# 以下の例では/workspace に data ディレクトリがあると想定
%cd /workspace/VQA

[Errno 2] No such file or directory: '/workspace/VQA'
/Users/daichi/dev-hub/kaggle/vqa


/Users/daichi/miniforge3/envs/my-default/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


### 1. import library

In [ ]:
import re
import random
import time

import numpy as np
import pandas as pd
import torch
import torchvision
import torch.nn as nn
from collections import Counter
from PIL import Image
from torchvision import transforms

### 2. utils

In [ ]:
def set_seed(seed):
    """
    シードを固定する．

    Parameters
    ----------
    seed : int
        乱数生成に用いるシード値．
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def process_text(text):
    """
    入力文と回答のフォーマットを統一するための関数．

    Parameters
    ----------
    text : str
        入力文，もしくは回答．
    """
    # lowercase
    text = text.lower()

    # 数詞を数字に変換
    num_word_to_digit = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'ten': '10'
    }
    for word, digit in num_word_to_digit.items():
        text = text.replace(word, digit)

    # 小数点のピリオドを削除
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)

    # 冠詞の削除
    text = re.sub(r'\b(a|an|the)\b', '', text)

    # 短縮形のカンマの追加
    contractions = {
        "dont": "don't", "isnt": "isn't", "arent": "aren't", "wont": "won't",
        "cant": "can't", "wouldnt": "wouldn't", "couldnt": "couldn't"
    }
    for contraction, correct in contractions.items():
        text = text.replace(contraction, correct)

    # 句読点をスペースに変換
    text = re.sub(r"[^\w\s':]", ' ', text)

    # 句読点をスペースに変換
    text = re.sub(r'\s+,', ',', text)

    # 連続するスペースを1つに変換
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
class VQADataset(torch.utils.data.Dataset):
    """
    VQA データセットを扱うためのクラス．
    """
    def __init__(self, df_path, image_dir, transform=None, answer=True, topk_answers=None):
        self.transform = transform  # 画像の前処理
        self.image_dir = image_dir  # 画像ファイルのディレクトリ
        self.df = pd.read_json(df_path)  # 画像ファイルのパス，question, answerを持つDataFrame
        self.answer = answer

        # question / answerの辞書を作成
        self.question2idx = {}
        self.answer2idx = {}
        self.idx2question = {}
        self.idx2answer = {}

        # 質問文に含まれる単語を辞書に追加
        for question in self.df["question"]:
            question = process_text(question)
            words = question.split(" ")
            for word in words:
                if word not in self.question2idx:
                    self.question2idx[word] = len(self.question2idx)
        self.idx2question = {v: k for k, v in self.question2idx.items()}  # 逆変換用の辞書(question)

        if self.answer:
            if topk_answers is None:
                # 回答に含まれる文章を辞書に追加
                for answers in self.df["answers"]:
                    for answer in answers:
                        word = answer["answer"]
                        word = process_text(word)
                        if word not in self.answer2idx:
                            self.answer2idx[word] = len(self.answer2idx)
                self.idx2answer = {v: k for k, v in self.answer2idx.items()}  # 逆変換用の辞書(answer)
            else:
                # Top-K のみ
                self.answer2idx = {a: i for i, a in enumerate(topk_answers)}
                self.idx2answer = {i: a for a, i in self.answer2idx.items()}

        self.n_answer = len(self.answer2idx)

    def update_dict(self, dataset):
        """
        検証用データ，テストデータの辞書を訓練データの辞書に更新する．

        Parameters
        ----------
        dataset : Dataset
            訓練データのDataset
        """
        self.question2idx = dataset.question2idx
        self.answer2idx = dataset.answer2idx
        self.idx2question = dataset.idx2question
        self.idx2answer = dataset.idx2answer

    def __getitem__(self, idx):
        """
        対応するidxのデータ（画像，質問，回答）を取得．

        Parameters
        ----------
        idx : int
            取得するデータのインデックス

        Returns
        -------
        image : torch.Tensor  (C, H, W)
            画像データ
        question : torch.Tensor  (vocab_size)
            質問文をone-hot表現に変換したもの
        answers : torch.Tensor  (n_answer)
            10人の回答者の回答のid
        mode_answer_idx : torch.Tensor  (1)
            10人の回答者の回答の中で最頻値の回答のid
        """
        image = Image.open(f"{self.image_dir}/{self.df['image'][idx]}").convert("RGB")
        image = self.transform(image)
        question = np.zeros(len(self.idx2question) + 1)  # 未知語用の要素を追加
        question_text = process_text(self.df["question"][idx])
        question_words = question_text.split()
        # for word in question_words:
        #     try:
        #         question[self.question2idx[word]] = 1  # one-hot表現に変換
        #     except KeyError:
        #         question[-1] = 1  # 未知語

        # if self.answer:
        #     answers = [self.answer2idx[process_text(answer["answer"])] for answer in self.df["answers"][idx]]
        #     mode_answer_idx = mode(answers)  # 最頻値を取得（正解ラベル）

        #     return image, torch.Tensor(question), torch.Tensor(answers), int(mode_answer_idx)

        # else:
        #     return image, torch.Tensor(question)
        for word in question_words:
            question[self.question2idx.get(word, len(self.idx2question))] = 1

        if self.answer:
            processed = [process_text(a["answer"]) for a in self.df["answers"][idx]]
            c = Counter(processed)

            target = torch.zeros(self.n_answer, dtype=torch.float32)
            for ans, cc in c.items():
                if ans in self.answer2idx:
                    target[self.answer2idx[ans]] = min(cc / 3.0, 1.0)

            return image, torch.tensor(question, dtype=torch.float32), processed, target

        else:
            return image, torch.tensor(question, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

In [ ]:
def VQA_criterion(batch_pred, batch_answers):
    """
    VQA タスクに用いられる評価関数．
    """
    total_acc = 0.

    for pred, answers in zip(batch_pred, batch_answers):
        acc = 0.
        for i in range(len(answers)):
            num_match = 0
            for j in range(len(answers)):
                if i == j:
                    continue
                if pred == answers[j]:
                    num_match += 1
            acc += min(num_match / 3, 1)
        total_acc += acc / 10

    return total_acc / len(batch_pred)

## 3. EDA

In [60]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
train_dataset = VQADataset(df_path="./data/train.json", image_dir="./data/train", transform=transform)
test_dataset = VQADataset(df_path="./data/valid.json", image_dir="./data/valid", transform=transform, answer=False)
test_dataset.update_dict(train_dataset)

In [61]:
train_dataset.df.head()

,image,question,answers
0,train_00000.jpg,What is this?,"[{'answer_confidence': 'yes', 'answer': 'beef ..."
1,train_00001.jpg,maybe it's because you're pushing it down instead,"[{'answer_confidence': 'yes', 'answer': 'unans..."
2,train_00002.jpg,What color is this item?,"[{'answer_confidence': 'yes', 'answer': 'grey'..."
3,train_00003.jpg,Can you tell me if this is like body wash or l...,"[{'answer_confidence': 'maybe', 'answer': 'lot..."
4,train_00004.jpg,Is it a paper?,"[{'answer_confidence': 'yes', 'answer': 'no'},..."


In [62]:
test_dataset.df.head()

,image,question
0,valid_00000.jpg,Was I able to clear either of the mirrors of t...
1,valid_00001.jpg,What page number is this above? Thank you.
2,valid_00002.jpg,Please tell me what is in this box.
3,valid_00003.jpg,Are the lights on in this room?
4,valid_00004.jpg,"What color is this? Please, thank you."


In [63]:
print(f"回答クラス数 n_answer: {len(train_dataset.answer2idx)}")
print(f"質問語彙数 bocab_size: {len(train_dataset.question2idx)}")

回答クラス数 n_answer: 40244
質問語彙数 bocab_size: 3908


分類するべきクラス数が 40,244 種類と非常に多いことがわかる

次に
- 10人回答全ての頻度 Top20
- mode (最頻値) だけの頻度 Top20
を見る

In [64]:
def safe_mode(items):
    # ties が合っても落ちない mode (最大頻度が複数あっても1つを返す)
    return Counter(items).most_common(1)[0][0]

# train_dataset.df は pandas.read_json で読んだ元データ
df = train_dataset.df

# (A) 10人回答全て (19,873 * 10 = 198,730 件) の頻度
all_answers = []
# (B) mode (各サンプル1つ、合計 19,873 件) の頻度
mode_answers = []

for answers in df["answers"]:
    processed = [process_text(a["answer"]) for a in answers]
    all_answers.extend(processed)
    mode_answers.append(safe_mode(processed))

cnt_all = Counter(all_answers)
cnt_mode = Counter(mode_answers)

def topk_table(counter, total, k=20):
    rows = []
    for ans, c in counter.most_common(k):
        rows.append({
            "answer": ans,
            "count": c,
            "pct(%)": 100.0 * c / total
        })
    return pd.DataFrame(rows)

top_all_df = topk_table(cnt_all, total=len(all_answers), k=20)
top_mode_df = topk_table(cnt_mode, total=len(mode_answers), k=20)

print(f"[A] 全回答 (10人×全サンプル) 件数: {len(all_answers)} / unique: {len(cnt_all)}")
print(f"[B] mode回答 (各サンプル1件) 件数: {len(mode_answers)} / unique: {len(cnt_mode)}")

print("\n[A] Top20 (全回答ベース)")
display(top_all_df)

print("\n[B] Top20 (modeベース)")
display(top_mode_df)

print("\n[A] Top20 coverage: ", top_all_df["count"].sum() / len(all_answers))
print("[B] Top20 coverage: ", top_mode_df["count"].sum() / len(mode_answers))

[A] 全回答 (10人×全サンプル) 件数: 198730 / unique: 40244
[B] mode回答 (各サンプル1件) 件数: 19873 / unique: 5314

[A] Top20 (全回答ベース)


,answer,count,pct(%)
0,unanswerable,55613,27.984200
1,no,5225,2.629195
2,yes,4337,2.182358
3,white,2511,1.263523
4,grey,2097,1.055201
5,black,2032,1.022493
6,blue,1716,0.863483
7,red,1087,0.546973
8,brown,787,0.396015
9,pink,748,0.376390



[B] Top20 (modeベース)


,answer,count,pct(%)
0,unanswerable,7559,38.036532
1,no,481,2.420369
2,yes,476,2.395210
3,white,300,1.509586
4,grey,266,1.338499
5,black,227,1.142253
6,blue,195,0.981231
7,red,115,0.578675
8,brown,99,0.498163
9,pink,91,0.457908



[A] Top20 coverage:  0.4088008856237106
[B] Top20 coverage:  0.5235746993408141


約4割のサンプルは「多数決で unanswerable」が正解ラベルになっていることがわかる

また、
- 10人回答を全部数えると、上位20語で約 41% を占める
- 「多数決ラベル」だけにすると、上位20語で 約52% を占める
ことがわかる

In [65]:
# 各サンプルの回答ID (10人分) を作る
all_answers_id = []
for answers in train_dataset.df["answers"]:
    processed = [process_text(a["answer"]) for a in answers]
    all_answers_id.append([train_dataset.answer2idx[p] for p in processed])

all_answers_id = torch.tensor(all_answers_id)  # (N, 10)

def vqa_acc_constant(pred_id: int):
    pred = torch.full((all_answers_id.size(0),), pred_id, dtype=torch.long)
    return VQA_criterion(pred, all_answers_id)

# 代表的な定数予測
for ans in ["unanswerable", "yes", "no", "white", "black"]:
    if ans in train_dataset.answer2idx:
        pid = train_dataset.answer2idx[ans]
        print(ans, vqa_acc_constant(pid))



unanswerable 0.4722538117043267
yes 0.03813213908317805
no 0.05443063452926017
white 0.023131887485533225
black 0.018447139334775917


- 常に `unanswerable` と予測するだけで、訓練データに対する VQA accuracy が約 0.472 に到達する
- 一方で、その他は 0.018=0.054 程度にしかならない
ことがわかる

In [66]:
def coverage_curve(counter, total, ks):
    # 上位kの累積カバー率
    freqs = np.array([c for _, c in counter.most_common()])
    cumsum = np.cumsum(freqs)
    out = []
    for k in ks:
        k = min(k, len(freqs))
        out.append(cumsum[k-1] / total)
    return out

ks = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

curve_all = coverage_curve(cnt_all, total=len(all_answers), ks=ks)
curve_mode = coverage_curve(cnt_mode, total=len(mode_answers), ks=ks)

df_curve = pd.DataFrame({
    "K": ks,
    "coverage_all_answers": curve_all,  # 10人回答全体
    "coverage_mode": curve_mode,  # mode のみ
})
df_curve

,K,coverage_all_answers,coverage_mode
0,1,0.279842,0.380365
1,2,0.306134,0.404569
2,5,0.351145,0.457002
3,10,0.383198,0.493584
4,20,0.408801,0.523575
5,50,0.445826,0.566397
6,100,0.481955,0.604740
7,200,0.522251,0.649927
8,500,0.582645,0.714588
9,1000,0.632758,0.772254


- mode に対しては 頻出上位 5000 語彙程度で足りる
- 10人回答 に対しては 5000 語彙でも 23% が対象外になる
つまり
- 出力クラスを 40,244 のままにするのは、mode 学習に対して非効率
- ただし 上位 K に絞りすぎると VQA (部分点) で不利

In [67]:
def score_from_c(c: int) -> float:
    if c <= 0:
        return 0.0
    if c == 1:
        return 0.3
    if c == 2:
        return 0.6
    if c == 3:
        return 0.9
    return 1.0

def oracle_vqa_acc_topk(train_df, K: int):
    topk_vocab = set([a for a, _ in cnt_all.most_common(K)])

    total = 0.0
    for answers in train_df["answers"]:
        processed = [process_text(a["answer"]) for a in answers]
        c = Counter(processed)

        # 上位K語彙に入っている回答のうち、最大一致数を取る
        max_c_in_topk = 0
        for ans, cc in c.items():
            if ans in topk_vocab:
                if cc > max_c_in_topk:
                    max_c_in_topk = cc

        total += score_from_c(max_c_in_topk)

    return total / len(train_df)

for K in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 40244]:
    print(K, oracle_vqa_acc_topk(train_dataset.df, K))


1 0.4722538117043267
2 0.48980526342273506
5 0.5370854928797818
10 0.5697579630654553
20 0.5986162129522332
50 0.6388466763950911
100 0.6751371207165285
200 0.7107482513963426
500 0.7649625119508596
1000 0.8039853066974998
2000 0.8483470034720183
5000 0.9170130327579957
10000 0.9395712776128352
40244 0.9415488351028986


分類する語彙を 10000 に絞っても全語彙 (40244) とほぼ変わらない上限性能を持っていることがわかる

次に、各サンプルで「同一回答の最大出現数」c_max (10人中で最多の回答が何票か) を数える。

In [68]:
def compute_cmax_stats(train_df):
    cmax_list = []
    uniq_list = []
    for answers in train_df["answers"]:
        processed = [process_text(a["answer"]) for a in answers]
        c = Counter(processed)
        cmax_list.append(max(c.values()))
        uniq_list.append(len(c))  # 10回答のユニーク数

    cmax = np.array(cmax_list)
    uniq = np.array(uniq_list)

    # 分布表 (票数ごとの割合)
    dist = pd.Series(cmax).value_counts().sort_index()
    dist = pd.DataFrame({
        "c_max": dist.index,
        "count": dist.values,
        "pct(%)": dist.values / len(cmax) * 100
    })

    # 参考: ユニーク数分布
    dist_u = pd.Series(uniq).value_counts().sort_index()
    dist_u = pd.DataFrame({
        "n_unique_in_10": dist_u.index,
        "count": dist_u.values,
        "pct(%)": dist_u.values / len(uniq) * 100
    })

    # まとめ統計
    summary = {
        "N": len(cmax),
        "cmax_mean": float(cmax.mean()),
        "cmax_median": float(np.median(cmax)),
        "pct_cmax_ge4": float((cmax >= 4).mean() * 100),
        "pct_cmax_eq3": float((cmax == 3).mean() * 100),
        "pct_cmax_eq2": float((cmax == 2).mean() * 100),
        "pct_cmax_eq1": float((cmax == 1).mean() * 100),
        "unique_mean": float(uniq.mean()),
        "unique_median": float(np.median(uniq)),
    }

    return dist, dist_u, summary

dist_cmax, dist_unique, summary = compute_cmax_stats(train_dataset.df)

print(summary)
display(dist_cmax)
display(dist_unique)

{'N': 19873, 'cmax_mean': 5.693855985507976, 'cmax_median': 6.0, 'pct_cmax_ge4': 77.73360841342524, 'pct_cmax_eq3': 12.252805313742265, 'pct_cmax_eq2': 7.965581442157702, 'pct_cmax_eq1': 2.048004830674785, 'unique_mean': 4.501031550344689, 'unique_median': 4.0}


,c_max,count,pct(%)
0,1,407,2.048005
1,2,1583,7.965581
2,3,2435,12.252805
3,4,2711,13.641624
4,5,2687,13.520857
5,6,2325,11.699290
6,7,2307,11.608715
7,8,2129,10.713028
8,9,2024,10.184673
9,10,1265,6.365420


,n_unique_in_10,count,pct(%)
0,1,1265,6.365420
1,2,2837,14.275650
2,3,3455,17.385397
3,4,3334,16.776531
4,5,2798,14.079404
5,6,2242,11.281638
6,7,1692,8.514064
7,8,1134,5.706235
8,9,709,3.567655
9,10,407,2.048005


## 5. model

In [69]:
class BasicBlock(nn.Module):
    """
    ResNet の basic block
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out


class BottleneckBlock(nn.Module):
    """
    ResNet の bottleneck block
    """
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, stride=1)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out


class ResNet(nn.Module):
    """
    ResNet の実装
    """
    def __init__(self, block, layers):
        """
        コンストラクタ．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        layers: list
            各ブロックの層数
        """
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, layers[0], 64)
        self.layer2 = self._make_layer(block, layers[1], 128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], 256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], 512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, 512)

    def _make_layer(self, block, blocks, out_channels, stride=1):
        """
        同じ構成を繰り返す部分を生成する．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        blocks: int
            層数
        out_channels: int
            出力のチャネル数
        stride: int
            ストライド

        Returns
        -------
        layers: torch.nn.ModuleList
            生成した層
        """
        layers = []
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            入力データ

        Returns
        -------
        x: torch.Tensor
            ResNet によって生成される特徴量
        """
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [70]:
def ResNet18():
    """
    ResNet18 を生成する関数．
    """
    return ResNet(BasicBlock, [2, 2, 2, 2])


def ResNet50():
    """
    ResNet50 を生成する関数．
    """
    return ResNet(BottleneckBlock, [3, 4, 6, 3])

In [71]:
class VQAModel(nn.Module):
    """
    VQA タスクを解くためのモデル例．
    """
    def __init__(self, vocab_size: int, n_answer: int):
        """
        コンストラクタ．

        Parameters
        ----------
        vocab_size: int
            入力文の語彙数
        n_answer: int
            出力のクラス数
        """
        super().__init__()
        self.resnet = ResNet18()
        self.text_encoder = nn.Linear(vocab_size, 512)

        self.fc = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, n_answer)
        )

    def forward(self, image, question):

        image_feature = self.resnet(image)  # 画像の特徴量
        question_feature = self.text_encoder(question)  # テキストの特徴量

        x = torch.cat([image_feature, question_feature], dim=1)
        x = self.fc(x)

        return x

## 6. train

In [72]:
# def train(model, dataloader, optimizer, criterion, device):
#     """
#     学習用の関数．

#     Parameters
#     ----------
#     model: torch.nn.Module
#         学習するモデル
#     dataloader: torch.utils.data.DataLoader
#         学習に利用するデータローダ
#     optimizer: torch.optim.Optim
#         最適化手法
#     criterion: torch.nn.Module
#         損失関数
#     device: torch.device
#         学習に利用するデバイス

#     Returns
#     -------
#     total_loss: float
#         平均損失
#     total_acc: float
#         平均正解率
#     simple_acc: float
#         最頻値に対する正解率（VQA の評価指標とは異なることに注意）
#     time: float
#         1 エポックの学習にかかった時間 (sec)
#     """
#     model.train()

#     total_loss = 0
#     total_acc = 0
#     simple_acc = 0

#     start = time.time()
#     for image, question, answers, mode_answer in dataloader:
#         image, question, answer, mode_answer = \
#             image.to(device), question.to(device), answers.to(device), mode_answer.to(device)

#         pred = model(image, question)
#         loss = criterion(pred, mode_answer.squeeze())

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#         total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
#         simple_acc += (pred.argmax(1) == mode_answer).float().mean().item()  # simple accuracy

#     return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start


# def eval(model, dataloader, optimizer, criterion, device):
#     """
#     学習用の関数．

#     Parameters
#     ----------
#     model: torch.nn.Module
#         モデル
#     dataloader: torch.utils.data.DataLoader
#         評価に利用するデータローダ
#     criterion: torch.nn.Module
#         損失関数
#     device: torch.device
#         利用するデバイス

#     Returns
#     -------
#     total_loss: float
#         平均損失
#     total_acc: float
#         平均正解率
#     simple_acc: float
#         最頻値に対する正解率（VQA の評価指標とは異なることに注意）
#     time: float
#         1 エポックの評価にかかった時間 (sec)
#     """
#     model.eval()

#     total_loss = 0
#     total_acc = 0
#     simple_acc = 0

#     start = time.time()
#     for image, question, answers, mode_answer in dataloader:
#         image, question, answer, mode_answer = \
#             image.to(device), question.to(device), answers.to(device), mode_answer.to(device)

#         pred = model(image, question)
#         loss = criterion(pred, mode_answer.squeeze())

#         total_loss += loss.item()
#         total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
#         simple_acc += (pred.argmax(1) == mode_answer).mean().item()  # simple accuracy

#     return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start

In [73]:
def train_soft(model, dataloader, optimizer, criterion, device, answer2idx):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    start = time.time()

    for image, question, answers_str, target in dataloader:
        image = image.to(device)
        question = question.to(device)
        target = target.to(device)  # (B, K)

        logits = model(image, question)  # (B, K)
        loss = criterion(logits, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # VQA accuracy (文字列ベース)
        pred_ids = logits.argmax(dim=1).detach().cpu().tolist()
        pred_str = [dataloader.dataset.idx2answer[i] for i in pred_ids]

        # answers_str は list[str] が batch 分入っている
        # dataloader のデフォルトcollateだと list[list[str]] になります
        # -> そのまま使えるように process_text 済みの answers_str を想定
        # VQA_criterion は pred と answers を "同じ型" で比較できればよい
        total_acc += VQA_criterion(pred_str, answers_str)

    n = len(dataloader)
    return total_loss / n, total_acc / n, time.time() - start

@torch.no_grad()
def eval_soft(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    start = time.time()

    for image, question, answers_str, target in dataloader:
        image = image.to(device)
        question = question.to(device)
        target = target.to(device)

        logits = model(image, question)
        loss = criterion(logits, target)
        total_loss += loss.item()

    n = len(dataloader)
    return total_loss / n, time.time() - start


## 7. make submission file

In [74]:
# deviceの設定
set_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# dataloader / model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Top-K answers を使って train_dataset を作る
K = 10000
topk_answers = [a for a, _ in cnt_all.most_common(K)]
topk_answers = [process_text(a) for a in topk_answers]

train_dataset = VQADataset(
    df_path="./data/train.json",
    image_dir="./data/train",
    transform=transform,
    answer=True,
    topk_answers=topk_answers
)
test_dataset = VQADataset(
    df_path="./data/valid.json",
    image_dir="./data/valid",
    transform=transform,
    answer=False
)
test_dataset.update_dict(train_dataset)

def collate_vqa_soft(batch):
    images, questions, answers_str, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    questions = torch.stack(questions, dim=0)
    targets = torch.stack(targets, dim=0)
    # answers_str は 「list[list[str]]」として保持 (転置しない)
    answers_str = list(answers_str)
    return images, questions, answers_str, targets

train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=128, 
    shuffle=True, 
    collate_fn=collate_vqa_soft
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

model = VQAModel(vocab_size=len(train_dataset.question2idx)+1, n_answer=len(train_dataset.answer2idx)).to(device)

# optimizer / criterion
num_epoch = 4
# criterion = nn.CrossEntropyLoss()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

Using device: mps


In [ ]:
# train model
for epoch in range(num_epoch):
    train_loss, train_acc, train_time = train_soft(model, train_loader, optimizer, criterion, device, train_dataset.answer2idx)
    print(f"【{epoch + 1}/{num_epoch}】\n"
            f"train time: {train_time:.2f} [s]\n"
            f"train loss: {train_loss:.4f}\n"
            f"train acc: {train_acc:.4f}\n")

In [ ]:
# make submission file
model.eval()
submission = []
for image, question in test_loader:
    image, question = image.to(device), question.to(device)
    pred = model(image, question)
    pred = pred.argmax(1).cpu().item()
    submission.append(pred)

submission = [train_dataset.idx2answer[id] for id in submission]
submission = np.array(submission)
torch.save(model.state_dict(), "model.pt")
np.save("submission.npy", submission)

## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (VQA)」から提出してください．

- `submission.npy`
- `model.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [ ]:
from zipfile import ZipFile

model_path = "model.pt"
notebook_path = "baseline.ipynb"

with ZipFile("submission.zip", "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path, arcname="baseline.ipynb")